# Ray Unit 4 Capstone - Codespaces Docker Launcher

Use this notebook in GitHub Codespaces for the live Docker-based Ray cluster demo. Codespaces is the Docker host, so there is no Docker Desktop and no separate VM setup in this flow.

This launcher verifies Docker, starts the course Docker Ray cluster, executes the three solution notebooks through Ray Jobs, and shows where the output artifacts are written.

## 1. Show Codespaces Runtime

This cell prints basic Linux runtime information from the Codespaces environment.

In [ ]:
!pwd
!uname -a
!python --version

## 2. Locate The Solution Directory

This cell finds the folder that contains `run_on_docker_engine.sh` and the three solution notebooks. It supports both the original `solution/` layout and the flattened Codespaces layout.

In [ ]:
from pathlib import Path
import os

start = Path.cwd()
solution_dir = None

def is_solution_folder(path: Path) -> bool:
    return all(
        (path / name).exists()
        for name in [
            "run_on_docker_engine.sh",
            "01_download_real_data.ipynb",
            "02_prepare_assets.ipynb",
            "03_run_replay.ipynb",
        ]
    )

for root in [start, *start.parents]:
    candidates = [
        root,
        root / "4_ray_capstone_project",
        root / "Ray" / "4_ray_capstone_project",
        root / "Ray" / "4_ray_capstone_project" / "solution",
    ]
    for candidate in candidates:
        if is_solution_folder(candidate):
            solution_dir = candidate
            break
    if solution_dir is not None:
        break

if solution_dir is None and Path("/workspaces").exists():
    matches = [
        path
        for pattern in [
            "*/4_ray_capstone_project",
            "*/Ray/4_ray_capstone_project",
            "*/Ray/4_ray_capstone_project/solution",
        ]
        for path in Path("/workspaces").glob(pattern)
        if is_solution_folder(path)
    ]
    if matches:
        solution_dir = matches[0]

if solution_dir is None:
    raise FileNotFoundError("Could not find the folder with run_on_docker_engine.sh and the three solution notebooks in this Codespace.")

cluster_candidates = [
    solution_dir / "../../1_cluster_setup",
    solution_dir / "../1_cluster_setup",
]
cluster_setup_dir = None
for candidate in cluster_candidates:
    resolved = candidate.resolve()
    if (resolved / "docker-compose.yml").exists():
        cluster_setup_dir = resolved
        break

if cluster_setup_dir is None:
    raise FileNotFoundError("Could not find 1_cluster_setup/docker-compose.yml from the solution folder.")

artifact_root = cluster_setup_dir / "head_workspace" / "ray_capstone"
os.chdir(solution_dir)
print("Working directory:", Path.cwd())
print("Cluster setup directory:", cluster_setup_dir)
print("Artifact root:", artifact_root)

## 3. Verify Docker In Codespaces

These commands are the Docker evidence for the live demo. They should show a reachable Docker Engine and Docker Compose command inside Codespaces.

In [ ]:
!docker version
!docker compose version || docker-compose version
!docker info --format 'Engine={{.ServerVersion}}; OS={{.OperatingSystem}}; OSType={{.OSType}}; CPUs={{.NCPU}}'

## 4. Run The Docker-Based Ray Cluster Flow

This starts the course virtual Docker cluster and executes the required notebook flow on Ray through Ray Jobs.

The helper script starts one `ray-head` container and the requested number of `ray-worker` containers. It then runs:

1. `01_download_real_data.ipynb`
2. `02_prepare_assets.ipynb`
3. `03_run_replay.ipynb`

Inside the Ray job, `RAY_ADDRESS=auto`, so the replay notebook connects to the Docker Ray cluster instead of starting isolated local Ray.

The helper executes the notebooks with `/opt/conda/envs/22971-ray/bin/python`, which is the Conda environment created from `environment.yml` inside the Docker image.

In [ ]:
!bash ./run_on_docker_engine.sh --workers 2

## 5. Ray Dashboard

After `ray-head` starts, Codespaces should forward port `8265`. Open the Codespaces **Ports** tab, find port `8265`, and open it in the browser.

The dashboard is useful during the video demo because it shows the Ray head, workers, jobs, and cluster activity.

In [ ]:
!cd {cluster_setup_dir} && (docker compose ps || docker-compose ps)
!docker ps --filter "name=ray" --format 'table {{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}'

## 6. Inspect Output Artifacts

The Docker run writes persistent files under the head container's mounted workspace. This folder remains visible in Codespaces after the Ray job finishes.

In [ ]:
!find {artifact_root} -maxdepth 3 -type f | sort | head -80
!echo ''
!echo 'Demo talking points:'
!cat {artifact_root}/outputs/demo_talking_points.md
!echo ''
!echo 'Demo summary CSV:'
!head -20 {artifact_root}/outputs/demo_summary.csv

## 7. Stop The Cluster After The Demo

Run this cleanup cell only after you finish inspecting the dashboard and artifacts.

In [ ]:
# Uncomment after the demo if you want to stop the containers.
# !cd {cluster_setup_dir} && (docker compose down || docker-compose down)